In [7]:
import nibabel as nib
import numpy as np
import os
import glob
import gc
import pandas as pd
from scipy.ndimage import generate_binary_structure, binary_erosion, binary_dilation, label
from skimage.morphology import convex_hull_image, remove_small_objects

# ==========================================
# БЛОК 1: МЕТРИКИ
# ==========================================
def compute_metrics(pred_mask, gt_mask, class_id):
    """
    Вычисляет IoU, Precision и Recall для заданного класса.
    """
    p = (pred_mask == class_id)
    g = (gt_mask == class_id)
    
    tp = np.logical_and(p, g).sum()
    fp = np.logical_and(p, ~g).sum()
    fn = np.logical_and(~p, g).sum()
    
    union = tp + fp + fn
    
    iou = tp / union if union > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    
    return iou, precision, recall

# ==========================================
# БЛОК 2: ТОПОЛОГИЧЕСКАЯ И КОНТЕКСТНАЯ ФИЛЬТРАЦИЯ
# ==========================================
def clean_noise_from_class(pred_data, class_id, min_size=50):
    """Удаляет мелкий изолированный мусор (соль) для заданного класса."""
    class_mask = (pred_data == class_id)
    clean_class_mask = remove_small_objects(class_mask, min_size=min_size)
    pred_data[class_mask] = 0
    pred_data[clean_class_mask] = class_id
    return pred_data

def get_largest_component(binary_mask):
    """Оставляет только самую большую связную область."""
    labeled_mask, num_features = label(binary_mask)
    if num_features == 0:
        return binary_mask
    
    volumes = np.bincount(labeled_mask.ravel())
    volumes[0] = 0
    largest_label = volumes.argmax()
    
    return labeled_mask == largest_label

def create_solid_hull_and_filter(pred_data, max_distance=5):
    """
    ЭТАП 1: Изолирует наполнитель и строит вокруг него ограничивающий 3D-кокон.
    """
    raw_filler_mask = (pred_data == 2)
    clean_filler = get_largest_component(raw_filler_mask)
    
    # Сразу обновляем класс наполнителя в предсказаниях — оставляем только монолит
    pred_data[raw_filler_mask] = 0
    pred_data[clean_filler] = 2
    
    del raw_filler_mask
    gc.collect()
    
    # Строим 3D Visual Hull пересечением оргональных проекций
    solid_3d = np.ones_like(clean_filler, dtype=bool)
    
    for z in range(clean_filler.shape[0]):
        if np.any(clean_filler[z]):
            solid_3d[z] &= convex_hull_image(clean_filler[z])
        else:
            solid_3d[z] = False
            
    for y in range(clean_filler.shape[1]):
        if np.any(clean_filler[:, y, :]):
            solid_3d[:, y, :] &= convex_hull_image(clean_filler[:, y, :])
        else:
            solid_3d[:, y, :] = False
            
    for x in range(clean_filler.shape[2]):
        if np.any(clean_filler[:, :, x]):
            solid_3d[:, :, x] &= convex_hull_image(clean_filler[:, :, x])
        else:
            solid_3d[:, :, x] = False
            
    del clean_filler
    gc.collect()
    
    # Расширяем кокон на max_distance для безопасного сохранения стенок банки
    struct = generate_binary_structure(3, 1)
    expanded_solid = binary_dilation(solid_3d, structure=struct, iterations=max_distance)
    
    del solid_3d
    gc.collect()
    
    # Жестко отсекаем всё внешнее пространство КТ-куба наружу от кокона
    pred_data[~expanded_solid] = 0
    
    del expanded_solid
    gc.collect()
    
    return pred_data

def remove_non_touching_components(pred_data, class_id, anchor_class_id=2, dilation_iters=2):
    """
    ЭТАП 2: Находит связные компоненты класса class_id и удаляет те из них,
    которые не имеют физического контакта с эталонным наполнителем (anchor_class_id).
    Memory-Safe оптимизация.
    """
    class_mask = (pred_data == class_id)
    anchor_mask = (pred_data == anchor_class_id)
    
    labeled_mask, num_features = label(class_mask)
    if num_features == 0:
        return pred_data
        
    # Слегка расширяем наполнитель булевой дилатацией, чтобы гарантированно поймать контакт по границе вокселей
    struct = generate_binary_structure(3, 1)
    dilated_anchor = binary_dilation(anchor_mask, structure=struct, iterations=dilation_iters)
    
    # Выделяем маску пересечения (только воксели стыка) — это экономит гигабайты ОЗУ
    touching_pixels_mask = (labeled_mask > 0) & dilated_anchor
    
    # Извлекаем метки только тех компонентов, которые коснулись наполнителя
    touching_labels = np.unique(labeled_mask[touching_pixels_mask])
    
    # Формируем булевую маску легитимных объектов
    keep_mask = np.isin(labeled_mask, touching_labels)
    
    # Удаляем нелегальные компоненты (например, корку на крышке), которые не коснулись наполнителя
    pred_data[class_mask & ~keep_mask] = 0
    
    del class_mask, anchor_mask, labeled_mask, dilated_anchor, touching_pixels_mask, keep_mask
    gc.collect()
    
    return pred_data

# ==========================================
# БЛОК 3: ОСНОВНОЙ ПАЙПЛАЙН ОБРАБОТКИ
# ==========================================
def evaluate_and_save_all(raw_pred_path, raw_gt_path, raw_image_path, 
                          eroded_gt_output_path, eroded_image_output_path, smart_clean_output_path, 
                          erosion_iters=33):
    
    pred_img = nib.load(raw_pred_path)
    pred_data = np.array(pred_img.dataobj, dtype=np.uint8)
    
    # --- ШАГ 1: Базовая чистка мелкой пыли ---
    pred_data = clean_noise_from_class(pred_data, class_id=1, min_size=500)  # Банка
    pred_data = clean_noise_from_class(pred_data, class_id=3, min_size=150)  # Предметы
    
    # --- ШАГ 2: Фиксация и отсечение наполнителя (3D Кокон) ---
    pred_data = create_solid_hull_and_filter(pred_data, max_distance=40)
    
    # --- ШАГ 3: Контекстное удаление компонентов, не касающихся наполнителя ---
    # Чистим красную корку предметов (Класс 3), налипшую на срезах КТ вне контакта с наполнителем
    pred_data = remove_non_touching_components(pred_data, class_id=3, anchor_class_id=2, dilation_iters=2)
    
    # --- ШАГ 4: Загрузка и обрезка GT / Снимка по FOV томографа ---
    gt_img = nib.load(raw_gt_path)
    gt_data = np.array(gt_img.dataobj, dtype=np.uint8) 
    
    if os.path.exists(raw_image_path):
        raw_img = nib.load(raw_image_path)
        raw_data = np.array(raw_img.dataobj)
        
        pad_value = raw_data[0, 0, 0]
        fov_mask = (raw_data != pad_value)
        eroded_fov = binary_erosion(fov_mask, structure=generate_binary_structure(3, 1), iterations=erosion_iters)
        del fov_mask
        
        gt_data[~eroded_fov] = 0
        pred_data[~eroded_fov] = 0  
        raw_data[~eroded_fov] = pad_value
        del eroded_fov

        eroded_raw_img = nib.Nifti1Image(raw_data, raw_img.affine, raw_img.header)
        nib.save(eroded_raw_img, eroded_image_output_path)
        del raw_data, eroded_raw_img
        gc.collect()
    else:
        print(f"[ВНИМАНИЕ] Сырой КТ не найден. Сохраняем GT без обрезки краев.")

    # Сохраняем результаты
    clean_img = nib.Nifti1Image(pred_data, pred_img.affine, pred_img.header)
    nib.save(clean_img, smart_clean_output_path)
    del clean_img

    eroded_gt_img = nib.Nifti1Image(gt_data, gt_img.affine, gt_img.header)
    nib.save(eroded_gt_img, eroded_gt_output_path)

    # Расчет финальных честных метрик
    metrics_1 = compute_metrics(pred_data, gt_data, class_id=1) 
    metrics_2 = compute_metrics(pred_data, gt_data, class_id=2) 
    metrics_3 = compute_metrics(pred_data, gt_data, class_id=3) 
    
    del pred_data, gt_data
    gc.collect()
    
    return metrics_1, metrics_2, metrics_3

# ==========================================
# ЗАПУСК И ГЕНЕРАЦИЯ ОТЧЕТА
# ==========================================
if __name__ == "__main__":
    raw_preds_dir   = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\predictionsTs_SO3" 
    raw_gt_dir      = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\labelsTs_SO3"
    images_dir      = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\imagesTs_SO3"
    
    eroded_gt_dir     = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\labelsTs_SO3_eroded"
    eroded_images_dir = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\imagesTs_SO3_eroded"
    smart_clean_dir   = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\_smart_cleaned_masks" 
    report_path       = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\metrics_full_report.csv"
    
    os.makedirs(eroded_gt_dir, exist_ok=True)
    os.makedirs(eroded_images_dir, exist_ok=True)
    os.makedirs(smart_clean_dir, exist_ok=True)
    
    results = []
    raw_files = glob.glob(os.path.join(raw_preds_dir, "*.nii.gz"))
    
    print(f"Генерация эталонных файлов визуализации. Файлов: {len(raw_files)}")
    print("-" * 65)
    
    for pred_path in raw_files:
        filename = os.path.basename(pred_path)
        raw_gt_path = os.path.join(raw_gt_dir, filename)
        
        raw_filename = filename.replace(".nii.gz", "_0000.nii.gz")
        raw_image_path = os.path.join(images_dir, raw_filename)
        
        eroded_gt_output_path = os.path.join(eroded_gt_dir, filename) 
        eroded_image_output_path = os.path.join(eroded_images_dir, raw_filename)
        smart_clean_output_path = os.path.join(smart_clean_dir, filename)
        
        if not os.path.exists(raw_gt_path):
            print(f"Пропуск {filename}: оригинальная разметка не найдена.")
            continue
            
        print(f"Обработка снимка: {filename}")
        
        (iou1, prec1, rec1), (iou2, prec2, rec2), (iou3, prec3, rec3) = evaluate_and_save_all(
            raw_pred_path=pred_path, 
            raw_gt_path=raw_gt_path, 
            raw_image_path=raw_image_path, 
            eroded_gt_output_path=eroded_gt_output_path,
            eroded_image_output_path=eroded_image_output_path,
            smart_clean_output_path=smart_clean_output_path,
            erosion_iters=33
        )
        
        results.append({
            "Filename": filename,
            "IoU_Cardboard": iou1, "Prec_Cardboard": prec1, "Rec_Cardboard": rec1,
            "IoU_Filler": iou2, "Prec_Filler": prec2, "Rec_Filler": rec2,
            "IoU_Objects": iou3, "Prec_Objects": prec3, "Rec_Objects": rec3
        })
        
    df = pd.DataFrame(results)
    
    print("\n" + "=" * 65)
    print("ИТОГОВЫЕ СРЕДНИЕ МЕТРИКИ (IoU | Precision | Recall):")
    print("-" * 65)
    print(f"Класс 1 (Банка):       {df['IoU_Cardboard'].mean():.4f} | {df['Prec_Cardboard'].mean():.4f} | {df['Rec_Cardboard'].mean():.4f}")
    print(f"Класс 2 (Наполнитель): {df['IoU_Filler'].mean():.4f} | {df['Prec_Filler'].mean():.4f} | {df['Rec_Filler'].mean():.4f}")
    print(f"Класс 3 (Предметы):    {df['IoU_Objects'].mean():.4f} | {df['Prec_Objects'].mean():.4f} | {df['Rec_Objects'].mean():.4f}")
    print("=" * 65)
    
    df.to_csv(report_path, index=False, sep=";")
    print(f"Полный отчет сохранен: {report_path}")

Генерация эталонных файлов визуализации. Файлов: 10
-----------------------------------------------------------------
Обработка снимка: banka_rot_000.nii.gz


C:\Users\semen\AppData\Local\Temp\ipykernel_96652\3071253349.py:150: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pred_data = np.array(pred_img.dataobj, dtype=np.uint8)
C:\Users\semen\AppData\Local\Temp\ipykernel_96652\3071253349.py:165: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  gt_data = np.array(gt_img.dataobj, dtype=np.uint8)
C:\Users\semen\AppData\Local\Temp\ipykernel_96652\3071253349.py:169: DeprecationWarning: __array__ implementation doesn't accept a copy keywor

Обработка снимка: banka_rot_001.nii.gz
Обработка снимка: banka_rot_002.nii.gz
Обработка снимка: banka_rot_003.nii.gz
Обработка снимка: banka_rot_004.nii.gz
Обработка снимка: banka_rot_005.nii.gz
Обработка снимка: banka_rot_006.nii.gz
Обработка снимка: banka_rot_007.nii.gz
Обработка снимка: banka_rot_008.nii.gz
Обработка снимка: banka_rot_009.nii.gz

ИТОГОВЫЕ СРЕДНИЕ МЕТРИКИ (IoU | Precision | Recall):
-----------------------------------------------------------------
Класс 1 (Банка):       0.7024 | 0.8884 | 0.7663
Класс 2 (Наполнитель): 0.8036 | 0.9721 | 0.8236
Класс 3 (Предметы):    0.6970 | 0.7211 | 0.9536
Полный отчет сохранен: C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\metrics_full_report.csv
